In [1]:
import os, json, base64, hashlib
import pandas as pd

os.chdir("/scratch/jq2uw/derm_vlms")

VLMS = {
    "DermatoLlama": "results/dermato_llama_predictions_paired.csv",
    "MedGemma": "results/medgemma_predictions_paired.csv",
    "GPT-5.3": "results/gpt53_predictions_paired.csv",
}
IMAGE_DIR = "results/images"
OUTPUT_PATH = "results/interface4/index.html"

QUESTIONS = ["Is the AI’s assessment accurate for this case? If not, what is incorrect and what should it say instead?"]

# --- load CSV data, keep only combined images and describe_then_classify ---
vlm_data = {}
all_ids = set()
for name, path in VLMS.items():
    if not os.path.exists(path):
        print(f"[SKIP] {name}: {path} not found (predictions may still be running)")
        continue
    df = pd.read_csv(path)
    df = df[df["id"].str.endswith("_combined")]
    if df.empty:
        print(f"[SKIP] {name}: no combined rows yet")
        continue
    records = df[["id", "describe_then_classify"]].to_dict("records")
    vlm_data[name] = records
    all_ids.update(df["id"])

# --- encode images as base64 ---
images = {}
for img_id in sorted(all_ids):
    p = os.path.join(IMAGE_DIR, f"{img_id}.jpg")
    with open(p, "rb") as f:
        images[img_id] = "data:image/jpeg;base64," + base64.b64encode(f.read()).decode()

fingerprint_src = json.dumps(QUESTIONS) + json.dumps(sorted(all_ids))
CONFIG_HASH = hashlib.md5(fingerprint_src.encode()).hexdigest()[:8]

print(f"Loaded {len(vlm_data)} VLMs, {len(images)} images  (config hash: {CONFIG_HASH})")

Loaded 3 VLMs, 10 images  (config hash: feabceb5)


In [2]:
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Dermatology VLM Evaluation</title>
<style>
*{margin:0;padding:0;box-sizing:border-box}
body{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;background:#f0f2f5;color:#1a1a2e;min-height:100vh}

header{background:#fff;border-bottom:1px solid #e0e0e0;padding:14px 24px;display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:10px;position:sticky;top:0;z-index:100}
header h1{font-size:1.2rem;color:#1a1a2e}
.hctl{display:flex;align-items:center;gap:14px;flex-wrap:wrap}
.progress{font-size:.9rem;color:#555;font-weight:500}
.btn-exp{padding:6px 14px;background:#059669;color:#fff;border:none;border-radius:6px;cursor:pointer;font-size:.85rem}
.btn-exp:hover{background:#047857}

main{max-width:1440px;margin:0 auto;padding:16px}
.panels{display:flex;gap:20px;align-items:flex-start}
.left-panel{flex:0 0 46%;position:sticky;top:70px}
.right-panel{flex:1;min-width:0}

.img-viewer{background:#fff;border-radius:12px;box-shadow:0 1px 3px rgba(0,0,0,.08);overflow:hidden}
.zoom-bar{display:flex;align-items:center;gap:6px;padding:8px 12px;border-bottom:1px solid #e5e7eb;background:#f9fafb}
.zoom-bar button{width:30px;height:30px;border:1px solid #d1d5db;background:#fff;border-radius:6px;cursor:pointer;font-size:1rem;display:flex;align-items:center;justify-content:center;color:#374151;transition:all .15s}
.zoom-bar button:hover{background:#f3f4f6;border-color:#9ca3af}
.zoom-bar button.active{background:#2563eb;color:#fff;border-color:#2563eb}
.zoom-bar .zlabel{font-size:.8rem;color:#6b7280;min-width:42px;text-align:center}
.zoom-bar .spacer{flex:1}
.img-container{position:relative;overflow:hidden;cursor:grab;background:#e5e7eb;max-height:85vh}
.img-container.crop-active{cursor:crosshair}
.img-container.panning{cursor:grabbing}
.img-container img{position:absolute;top:0;left:0;transform-origin:0 0;user-select:none;-webkit-user-drag:none}
.crop-overlay{position:absolute;inset:0;display:none;z-index:10}
.crop-sel{position:absolute;border:2px solid #2563eb;background:rgba(37,99,235,.12);pointer-events:none}
.img-id{padding:6px 12px;font-size:.82rem;color:#888;text-align:center;background:#f9fafb;border-top:1px solid #e5e7eb}

.task-sec{background:#fff;border-radius:12px;box-shadow:0 1px 3px rgba(0,0,0,.08);margin-bottom:18px;overflow:hidden}
.tabs{display:flex;border-bottom:2px solid #e5e7eb}
.tab{flex:1;padding:11px;text-align:center;background:#f9fafb;border:none;cursor:pointer;font-size:.9rem;font-weight:500;color:#6b7280;transition:all .2s}
.tab:hover{background:#f3f4f6}
.tab.active{background:#fff;color:#2563eb;border-bottom:2px solid #2563eb;margin-bottom:-2px}
.tab .dot{display:inline-block;width:7px;height:7px;border-radius:50%;margin-left:6px;background:#d1d5db;vertical-align:middle}
.tab .dot.done{background:#059669}

.tc{padding:20px 24px}
.field{margin-bottom:16px}
.field-label{font-size:.78rem;text-transform:uppercase;letter-spacing:.05em;color:#6b7280;font-weight:600;margin-bottom:4px}
.field-val{font-size:.93rem;line-height:1.5;color:#1e293b}
.field-val.prompt-val{font-style:italic;color:#374151}
.resp{background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:14px;font-size:.91rem;line-height:1.6;color:#334155;max-height:220px;overflow-y:auto;white-space:pre-wrap}

.q-label{font-size:.9rem;font-weight:500;color:#374151;margin-bottom:8px}
.survey{display:flex;flex-direction:column;gap:10px}
.survey textarea{width:100%;padding:7px 10px;border:1px solid #d1d5db;border-radius:6px;font-size:.9rem;font-family:inherit;resize:vertical;min-height:180px;line-height:1.5}
.survey textarea:focus{outline:none;border-color:#2563eb;box-shadow:0 0 0 2px rgba(37,99,235,.15)}

.evidence-area{margin-top:10px}
.ev-title{font-size:.78rem;text-transform:uppercase;letter-spacing:.05em;color:#6b7280;font-weight:600;margin-bottom:6px}
.evidence-list{display:flex;flex-wrap:wrap;gap:8px}
.evidence-item{position:relative;border:1px solid #d1d5db;border-radius:6px;overflow:hidden;background:#f8fafc}
.evidence-item img{display:block;max-width:100px;max-height:80px;object-fit:contain}
.evidence-item .ev-label{font-size:.7rem;color:#6b7280;text-align:center;padding:2px 4px;background:#f1f5f9}
.evidence-item .ev-remove{position:absolute;top:2px;right:2px;width:18px;height:18px;border-radius:50%;border:none;background:rgba(239,68,68,.85);color:#fff;font-size:.75rem;cursor:pointer;display:flex;align-items:center;justify-content:center;line-height:1;opacity:0;transition:opacity .15s}
.evidence-item:hover .ev-remove{opacity:1}

.btn-row{display:flex;align-items:center;gap:8px;margin-top:12px;flex-wrap:wrap}
.btn-copy{display:inline-flex;align-items:center;gap:5px;padding:5px 12px;background:#eef2ff;color:#4338ca;border:1px solid #c7d2fe;border-radius:6px;cursor:pointer;font-size:.8rem;font-weight:500;transition:all .15s}
.btn-copy:hover{background:#e0e7ff;border-color:#a5b4fc}

.nav{display:flex;justify-content:center;align-items:center;gap:14px;padding:18px 0 36px}
.nav button{padding:9px 26px;border-radius:8px;font-size:.93rem;cursor:pointer;font-weight:500;border:1px solid #d1d5db;background:#fff;color:#374151;transition:all .15s}
.nav button:hover:not(:disabled){background:#f3f4f6}
.nav button:disabled{opacity:.35;cursor:not-allowed}
.status{font-size:.8rem;color:#059669;font-weight:500;min-width:80px;text-align:center;opacity:0;transition:opacity .3s}
.status.show{opacity:1}

@media(max-width:960px){
  .panels{flex-direction:column}
  .left-panel{flex:none;width:100%;position:static}
  .img-container{max-height:60vh}
}
</style>
</head>
<body>

<header>
  <h1>Dermatology VLM Evaluation</h1>
  <div class="hctl">
    <span class="progress" id="prog"></span>
    <button class="btn-exp" id="btn-exp">Export CSV</button>
  </div>
</header>

<main>
<div class="panels">

  <!-- LEFT: Image viewer with zoom & crop -->
  <div class="left-panel">
    <div class="img-viewer">
      <div class="zoom-bar">
        <button id="btn-zin" title="Zoom in">+</button>
        <span class="zlabel" id="zoom-lbl">100%</span>
        <button id="btn-zout" title="Zoom out">&minus;</button>
        <button id="btn-zreset" title="Reset zoom">&#x21BA;</button>
        <span class="spacer"></span>
        <span style="font-size:.72rem;color:#9ca3af;line-height:1.3;text-align:right">Use Crop to select a region of the image<br>as evidence for any part of your feedback</span>
        <button id="btn-crop" style="font-size:.78rem">&#x2702; Crop</button>
      </div>
      <div class="img-container" id="img-container">
        <img id="img" alt="lesion" />
        <div class="crop-overlay" id="crop-overlay">
          <div class="crop-sel" id="crop-sel"></div>
        </div>
      </div>
      <div class="img-id" id="img-id"></div>
    </div>
  </div>

  <!-- RIGHT: Response & feedback -->
  <div class="right-panel">
    <div class="task-sec">
      <div class="tabs" id="tabs"></div>
      <div class="tc">
        <div class="field">
          <div class="field-label">Prompt</div>
          <div class="field-val prompt-val">You are an expert dermatologist. Please describe using dermatological terms to describe the appearance of this lesion. Please give the top 3 diagnoses in your differential.</div>
        </div>
        <div class="field">
          <div class="field-label">VLM Response</div>
          <div class="resp" id="resp"></div>
        </div>
        <p class="q-label">Is the AI's assessment accurate for this case? If not, what is incorrect and what should it say instead?</p>
        <div class="survey" id="survey"></div>
        <div class="evidence-area" id="evidence-area" style="display:none">
          <div class="ev-title">Visual Evidence</div>
          <div class="evidence-list" id="evidence-list"></div>
        </div>
        <div class="btn-row">
          <button class="btn-copy" id="btn-copy">&#x270E; Edit AI Response</button>
          <span style="font-size:.78rem;color:#6b7280">(Copies response into text box for editing)</span>
        </div>
      </div>
    </div>

    <div class="nav">
      <button id="btn-prev">&laquo; Previous</button>
      <span class="status" id="status"></span>
      <button id="btn-next">Next &raquo;</button>
    </div>
  </div>

</div>
</main>

<script>
/* ===== DATA ===== */
const VLM_DATA  = __VLM_DATA__;
const IMAGES    = __IMAGES__;
const QUESTIONS = __QUESTIONS__;
const TASK_KEY  = 'describe_then_classify';
const SK = 'vlm_eval_dc___CONFIG_HASH__';

/* ===== STATE ===== */
const S = {
  vlm:  Object.keys(VLM_DATA)[0],
  idx:  0,
  R:    JSON.parse(localStorage.getItem(SK) || '{}'),
  zoom: 1, panX: 0, panY: 0,
  baseW: 0, baseH: 0,
  cropMode: false,
  isPanning: false,
  panStartX: 0, panStartY: 0,
  panOrigX: 0, panOrigY: 0
};

/* ===== ELEMENTS ===== */
const imgEl       = document.getElementById('img');
const containerEl = document.getElementById('img-container');
const cropOverlay = document.getElementById('crop-overlay');
const cropSel     = document.getElementById('crop-sel');
const zoomLbl     = document.getElementById('zoom-lbl');
const evArea      = document.getElementById('evidence-area');
const evList      = document.getElementById('evidence-list');

/* ===== VLM TABS ===== */
const VLM_NAMES = Object.keys(VLM_DATA);
const tabsEl = document.getElementById('tabs');
VLM_NAMES.forEach(n => {
  const b = document.createElement('button');
  b.className = 'tab'; b.dataset.vlm = n;
  b.innerHTML = n + '<span class="dot"></span>';
  b.onclick = () => { collect(); S.vlm = n; render(); };
  tabsEl.appendChild(b);
});

/* ===== QUESTION INPUTS ===== */
const surveyEl = document.getElementById('survey');
QUESTIONS.forEach((q, i) => {
  const inp = document.createElement('textarea');
  inp.id = 'q' + i; inp.placeholder = 'Your response...'; inp.rows = 4;
  surveyEl.appendChild(inp);
});
const qInputs = QUESTIONS.map((_, i) => document.getElementById('q' + i));

/* ===== TEXT CLEANING ===== */
function cleanText(text) {
  if (!text) return '';
  let t = text;
  t = t.replace(/\*{3,}/g, '');
  t = t.replace(/\*\*(.+?)\*\*/gs, '$1');
  t = t.replace(/(?<!\*)\*(?!\*)(.+?)(?<!\*)\*(?!\*)/gs, '$1');
  t = t.replace(/\*/g, '');
  t = t.replace(/^#{1,6}\s*/gm, '');
  t = t.replace(/^[-_]{3,}\s*$/gm, '');
  t = t.replace(/\n{3,}/g, '\n\n');
  return t.trim();
}

/* ===== DATA HELPERS ===== */
function row() { return VLM_DATA[S.vlm][S.idx]; }
function rkey(i) { return TASK_KEY + '_q' + (i + 1); }
function getR(i) { return S.R[S.vlm]?.[row().id]?.[rkey(i)] || ''; }
function setR(i, v) {
  if (!S.R[S.vlm]) S.R[S.vlm] = {};
  if (!S.R[S.vlm][row().id]) S.R[S.vlm][row().id] = {};
  S.R[S.vlm][row().id][rkey(i)] = v;
}
function collect() { qInputs.forEach((el, i) => setR(i, el.value)); }
function persist() { localStorage.setItem(SK, JSON.stringify(S.R)); }

function getCrops() {
  return S.R[S.vlm]?.[row().id]?.crops || [];
}
function setCrops(crops) {
  if (!S.R[S.vlm]) S.R[S.vlm] = {};
  if (!S.R[S.vlm][row().id]) S.R[S.vlm][row().id] = {};
  S.R[S.vlm][row().id].crops = crops;
}

function fmtId(id) {
  const m = id.match(/^(\d+)_combined$/);
  return m ? 'Lesion ' + m[1] : id;
}
function vlmDone(vlm) {
  const imgId = VLM_DATA[vlm][S.idx].id;
  const r = S.R[vlm]?.[imgId];
  if (!r) return false;
  return QUESTIONS.every((_, i) => {
    const v = r[TASK_KEY + '_q' + (i + 1)];
    return v && v.trim();
  });
}

/* ===== IMAGE FIT / TRANSFORM ===== */
function fitImage() {
  if (!imgEl.naturalWidth) return;
  const cw = containerEl.clientWidth;
  const nw = imgEl.naturalWidth, nh = imgEl.naturalHeight;
  const maxH = window.innerHeight * 0.85;
  let dw = cw, dh = cw * (nh / nw);
  if (dh > maxH) { dh = maxH; dw = maxH * (nw / nh); }
  containerEl.style.height = dh + 'px';
  S.baseW = dw;
  S.baseH = dh;
  updateTransform();
}
function updateTransform() {
  const cw = containerEl.clientWidth, ch = containerEl.clientHeight;
  const cx = (cw - S.baseW * S.zoom) / 2 + S.panX;
  const cy = (ch - S.baseH * S.zoom) / 2 + S.panY;
  imgEl.style.width  = S.baseW + 'px';
  imgEl.style.height = S.baseH + 'px';
  imgEl.style.transformOrigin = '0 0';
  imgEl.style.transform = 'translate(' + cx + 'px,' + cy + 'px) scale(' + S.zoom + ')';
  zoomLbl.textContent = Math.round(S.zoom * 100) + '%';
}
function resetView() {
  S.zoom = 1; S.panX = 0; S.panY = 0;
  updateTransform();
}

/* ===== ZOOM ===== */
function zoomBy(delta, cx, cy) {
  const oldZ = S.zoom;
  S.zoom = Math.min(Math.max(S.zoom * (1 + delta), 0.5), 8);
  if (cx !== undefined) {
    const cw = containerEl.clientWidth, ch = containerEl.clientHeight;
    const oldIx = (cw - S.baseW * oldZ) / 2 + S.panX;
    const oldIy = (ch - S.baseH * oldZ) / 2 + S.panY;
    const ix = (cx - oldIx) / oldZ;
    const iy = (cy - oldIy) / oldZ;
    const newIx = (cw - S.baseW * S.zoom) / 2 + S.panX;
    const newIy = (ch - S.baseH * S.zoom) / 2 + S.panY;
    S.panX += cx - (newIx + ix * S.zoom);
    S.panY += cy - (newIy + iy * S.zoom);
  }
  updateTransform();
}
containerEl.addEventListener('wheel', e => {
  e.preventDefault();
  const r = containerEl.getBoundingClientRect();
  zoomBy(e.deltaY < 0 ? 0.15 : -0.15, e.clientX - r.left, e.clientY - r.top);
}, { passive: false });
document.getElementById('btn-zin').onclick  = () => zoomBy(0.25);
document.getElementById('btn-zout').onclick = () => zoomBy(-0.25);
document.getElementById('btn-zreset').onclick = resetView;

/* ===== PAN (default mode) ===== */
containerEl.addEventListener('mousedown', e => {
  if (S.cropMode || e.button !== 0) return;
  S.isPanning = true;
  S.panStartX = e.clientX; S.panStartY = e.clientY;
  S.panOrigX = S.panX; S.panOrigY = S.panY;
  containerEl.classList.add('panning');
  e.preventDefault();
});
window.addEventListener('mousemove', e => {
  if (!S.isPanning) return;
  S.panX = S.panOrigX + (e.clientX - S.panStartX);
  S.panY = S.panOrigY + (e.clientY - S.panStartY);
  updateTransform();
});
window.addEventListener('mouseup', () => {
  if (S.isPanning) { S.isPanning = false; containerEl.classList.remove('panning'); }
});

/* ===== CROP MODE ===== */
const btnCrop = document.getElementById('btn-crop');
let cropStart = null;

btnCrop.onclick = () => {
  S.cropMode = !S.cropMode;
  btnCrop.classList.toggle('active', S.cropMode);
  cropOverlay.style.display = S.cropMode ? 'block' : 'none';
  containerEl.classList.toggle('crop-active', S.cropMode);
};

function screenToImgNorm(sx, sy) {
  const cw = containerEl.clientWidth, ch = containerEl.clientHeight;
  const imgCx = (cw - S.baseW * S.zoom) / 2 + S.panX;
  const imgCy = (ch - S.baseH * S.zoom) / 2 + S.panY;
  const ix = (sx - imgCx) / (S.baseW * S.zoom);
  const iy = (sy - imgCy) / (S.baseH * S.zoom);
  return { x: Math.max(0, Math.min(1, ix)), y: Math.max(0, Math.min(1, iy)) };
}

cropOverlay.addEventListener('mousedown', e => {
  if (!S.cropMode) return;
  e.preventDefault(); e.stopPropagation();
  const r = containerEl.getBoundingClientRect();
  cropStart = { sx: e.clientX - r.left, sy: e.clientY - r.top };
  cropSel.style.display = 'none';
});
cropOverlay.addEventListener('mousemove', e => {
  if (!cropStart) return;
  const r = containerEl.getBoundingClientRect();
  const mx = e.clientX - r.left, my = e.clientY - r.top;
  const x1 = Math.min(cropStart.sx, mx), y1 = Math.min(cropStart.sy, my);
  const x2 = Math.max(cropStart.sx, mx), y2 = Math.max(cropStart.sy, my);
  cropSel.style.display = 'block';
  cropSel.style.left = x1 + 'px'; cropSel.style.top = y1 + 'px';
  cropSel.style.width = (x2 - x1) + 'px'; cropSel.style.height = (y2 - y1) + 'px';
});
cropOverlay.addEventListener('mouseup', e => {
  if (!cropStart) return;
  const r = containerEl.getBoundingClientRect();
  const mx = e.clientX - r.left, my = e.clientY - r.top;
  const p1 = screenToImgNorm(cropStart.sx, cropStart.sy);
  const p2 = screenToImgNorm(mx, my);
  cropSel.style.display = 'none';
  cropStart = null;
  const x = Math.min(p1.x, p2.x), y = Math.min(p1.y, p2.y);
  const w = Math.abs(p2.x - p1.x), h = Math.abs(p2.y - p1.y);
  if (w < 0.01 || h < 0.01) return;
  addCropEvidence({ x, y, w, h });
  S.cropMode = false;
  btnCrop.classList.remove('active');
  cropOverlay.style.display = 'none';
  containerEl.classList.remove('crop-active');
});

/* ===== EVIDENCE MANAGEMENT ===== */
let cachedImg = null, cachedImgId = null;

function ensureCachedImg(cb) {
  const id = row().id;
  if (cachedImgId === id && cachedImg && cachedImg.complete) { cb(); return; }
  cachedImg = new Image();
  cachedImg.onload = () => { cachedImgId = id; cb(); };
  cachedImg.src = IMAGES[id];
}

function addCropEvidence(crop) {
  const crops = [...getCrops(), crop];
  setCrops(crops);
  persist();
  renderEvidence();
  const ta = qInputs[0];
  if (ta) {
    const marker = '[Evidence ' + crops.length + '] ';
    const pos = ta.selectionStart || ta.value.length;
    ta.value = ta.value.slice(0, pos) + marker + ta.value.slice(pos);
    ta.focus();
    ta.selectionStart = ta.selectionEnd = pos + marker.length;
    collect(); persist();
  }
  showSaved();
}

function removeCropEvidence(idx) {
  const oldLen = getCrops().length;
  const crops = getCrops().filter((_, i) => i !== idx);
  setCrops(crops);
  const ta = qInputs[0];
  if (ta) {
    let text = ta.value;
    const removed = idx + 1;
    text = text.replace(new RegExp('\\[Evidence\\s+' + removed + '\\]\\s?', 'g'), '');
    for (let n = removed + 1; n <= oldLen; n++) {
      text = text.replace(new RegExp('\\[Evidence\\s+' + n + '\\]', 'g'), '[Evidence ' + (n - 1) + ']');
    }
    ta.value = text;
    collect();
  }
  persist();
  renderEvidence();
}

function renderEvidence() {
  const crops = getCrops();
  evArea.style.display = crops.length ? '' : 'none';
  evList.innerHTML = '';
  if (!crops.length) return;
  ensureCachedImg(() => {
    const nw = cachedImg.naturalWidth, nh = cachedImg.naturalHeight;
    crops.forEach((c, idx) => {
      const sx = c.x * nw, sy = c.y * nh, sw = c.w * nw, sh = c.h * nh;
      const canvas = document.createElement('canvas');
      const sc = Math.min(100 / sw, 100 / sh, 1);
      canvas.width = Math.max(1, sw * sc); canvas.height = Math.max(1, sh * sc);
      canvas.getContext('2d').drawImage(cachedImg, sx, sy, sw, sh, 0, 0, canvas.width, canvas.height);
      const div = document.createElement('div'); div.className = 'evidence-item';
      const thumb = document.createElement('img'); thumb.src = canvas.toDataURL('image/jpeg', 0.85);
      const lbl = document.createElement('div'); lbl.className = 'ev-label';
      lbl.textContent = 'Evidence ' + (idx + 1);
      const rm = document.createElement('button'); rm.className = 'ev-remove';
      rm.innerHTML = '&times;';
      rm.onclick = () => { removeCropEvidence(idx); };
      div.appendChild(thumb); div.appendChild(lbl); div.appendChild(rm);
      evList.appendChild(div);
    });
  });
}

/* ===== EDIT AI RESPONSE ===== */
document.getElementById('btn-copy').onclick = () => {
  const resp = document.getElementById('resp').textContent;
  if (qInputs.length > 0) { qInputs[0].value = resp; qInputs[0].focus(); collect(); persist(); }
  showSaved('\u2713 Copied');
};

/* ===== STATUS ===== */
let saveTimer;
function showSaved(msg) {
  const el = document.getElementById('status');
  el.textContent = msg || '\u2713 Saved';
  el.classList.add('show');
  clearTimeout(saveTimer);
  saveTimer = setTimeout(() => el.classList.remove('show'), 1500);
}

/* ===== RENDER ===== */
function render() {
  const r = row(), data = VLM_DATA[S.vlm];
  imgEl.onload = fitImage;
  imgEl.src = IMAGES[r.id];
  resetView();
  document.getElementById('img-id').textContent = fmtId(r.id);
  document.getElementById('prog').textContent = 'Lesion ' + (S.idx + 1) + ' of ' + data.length;
  document.getElementById('btn-prev').disabled = S.idx === 0;
  document.getElementById('btn-next').disabled = S.idx === data.length - 1;
  document.querySelectorAll('.tab').forEach(t => {
    const v = t.dataset.vlm;
    t.classList.toggle('active', v === S.vlm);
    t.querySelector('.dot').classList.toggle('done', vlmDone(v));
  });
  document.getElementById('resp').textContent = cleanText(r[TASK_KEY]);
  qInputs.forEach((el, i) => { el.value = getR(i); });
  renderEvidence();
}

/* ===== NAVIGATION ===== */
document.getElementById('btn-prev').onclick = () => { collect(); persist(); if (S.idx > 0) { S.idx--; render(); } };
document.getElementById('btn-next').onclick = () => { collect(); persist(); if (S.idx < VLM_DATA[S.vlm].length - 1) { S.idx++; render(); } };

/* ===== SYNC: text edits remove crops ===== */
function syncCropsFromText() {
  const ta = qInputs[0];
  if (!ta) return;
  const crops = getCrops();
  if (!crops.length) return;
  const present = new Set();
  const re = /\[Evidence\s+(\d+)\]/g;
  let m;
  while ((m = re.exec(ta.value)) !== null) present.add(parseInt(m[1]));
  if (present.size === crops.length) return;
  const kept = [];
  const oldToNew = {};
  for (let i = 0; i < crops.length; i++) {
    if (present.has(i + 1)) { oldToNew[i + 1] = kept.length + 1; kept.push(crops[i]); }
  }
  setCrops(kept);
  let text = ta.value;
  for (const [old, nw] of Object.entries(oldToNew)) {
    if (+old !== nw) text = text.replace(new RegExp('\\[Evidence\\s+' + old + '\\]', 'g'), '[Evidence ' + nw + ']');
  }
  ta.value = text;
  collect(); persist();
  renderEvidence();
}

/* ===== AUTO-SAVE ===== */
document.getElementById('survey').addEventListener('input', () => {
  collect(); persist();
  syncCropsFromText();
  showSaved('\u2713 Auto-saved');
});

/* ===== EXPORT CSV ===== */
document.getElementById('btn-exp').onclick = () => {
  collect(); persist();
  const qHeaders = QUESTIONS.map((_, i) => 'Q' + (i + 1));
  const qKeys = QUESTIONS.map((_, i) => rkey(i));
  let csv = ['id', 'model', 'response', ...qHeaders, 'crop_coordinates'].join(',') + '\n';
  Object.keys(VLM_DATA).forEach(vlm => {
    const rows = VLM_DATA[vlm], resp = S.R[vlm] || {};
    rows.forEach(r => {
      const a = resp[r.id] || {};
      const crops = a.crops ? JSON.stringify(a.crops) : '';
      const vals = [r.id, vlm, r[TASK_KEY], ...qKeys.map(c => a[c] || ''), crops];
      csv += vals.map(v => '"' + String(v).replace(/"/g, '""') + '"').join(',') + '\n';
    });
  });
  const blob = new Blob([csv], { type: 'text/csv' });
  const link = document.createElement('a');
  link.href = URL.createObjectURL(blob);
  link.download = 'vlm_evaluation_responses.csv';
  document.body.appendChild(link); link.click(); document.body.removeChild(link);
};

/* ===== KEYBOARD ===== */
document.addEventListener('keydown', e => {
  if (e.target.tagName === 'INPUT' || e.target.tagName === 'TEXTAREA') return;
  if (e.key === 'ArrowLeft') document.getElementById('btn-prev').click();
  if (e.key === 'ArrowRight') document.getElementById('btn-next').click();
  if (e.key === 'Escape' && S.cropMode) {
    S.cropMode = false; btnCrop.classList.remove('active');
    cropOverlay.style.display = 'none'; containerEl.classList.remove('crop-active');
    cropStart = null; cropSel.style.display = 'none';
  }
});

window.addEventListener('resize', fitImage);
render();
</script>
</body>
</html>"""

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

html = (HTML_TEMPLATE
    .replace("__VLM_DATA__", json.dumps(vlm_data))
    .replace("__IMAGES__",   json.dumps(images))
    .replace("__QUESTIONS__", json.dumps(QUESTIONS))
    .replace("__CONFIG_HASH__", CONFIG_HASH)
)

with open(OUTPUT_PATH, "w") as f:
    f.write(html)

size_mb = os.path.getsize(OUTPUT_PATH) / 1024 / 1024
print(f"Generated {OUTPUT_PATH} ({size_mb:.1f} MB)")

Generated results/interface4/index.html (63.8 MB)
